In [ ]:
import sys

sys.path.append("../../..")
from setup_figs import clean_names, pd, sns, np, plt, stats

In [ ]:
fi = pd.read_parquet("0.parquet")
fi = clean_names(fi[fi.split == "test"])
spearman = pd.read_parquet("1.parquet")
spearman = clean_names(spearman[spearman.split == "test"])
weights = clean_names(pd.read_parquet("2.parquet"))
weights["AbsWeight"] = weights["Weight"].abs()

In [ ]:
id_cols = ["trainer.model_builder.param", "trainer.representations.layer", "cv"]
hue_order = ["MLEM", "FR-RSA-I"]

# Spearman

In [ ]:
ax = sns.lineplot(
    spearman,
    x="trainer.representations.layer",
    y="mean",
    hue="trainer.model_builder.param",
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_xlabel("Layer")
ax.set_ylabel("Spearman")
ax.get_legend().set_title(None)
ax.set_xticks([1, 3, 6, 9, 12])
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/layers/spearman.pdf", bbox_inches="tight")
plt.show()

# Training duration

In [ ]:
ax = sns.lineplot(
    weights,
    x="trainer.representations.layer",
    y="training_duration",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.set_xlabel("Layer")
ax.set_ylabel("Training duration (s)")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
sns.move_legend(ax, "upper right", bbox_to_anchor=(1, 1.25), title=None)
ax.set_xticks([1, 3, 6, 9, 12])
sns.despine(trim=True)
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/layers/training_duration.pdf", bbox_inches="tight"
)
plt.show()

# FI

In [ ]:
features_fi = (
    fi.groupby(["Feature", "trainer.model_builder.param"])["mean"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(4)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features_weights = (
    weights.groupby(["Feature", "trainer.model_builder.param"])
    .AbsWeight.mean()
    .sort_values(ascending=False)
    .reset_index()
    .groupby("trainer.model_builder.param")
    .head(4)[["Feature", "trainer.model_builder.param"]]
)

In [ ]:
features = pd.concat([features_fi, features_weights])[["Feature"]].drop_duplicates()
features

In [ ]:
g = sns.relplot(
    fi.merge(features),
    kind="line",
    x="trainer.representations.layer",
    y="mean",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    col_order=hue_order,
    height=4.5,
    aspect=1.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.set_xlabel("Layer")
    ax.set_ylabel("Feature Importance")
g.fig.subplots_adjust(wspace=0.05)
ax.set_xticks([1, 3, 6, 9, 12])
sns.despine(trim=True)
plt.savefig(
    "../../../paper/figs/BERT_RC_mean/layers/feature_importance.pdf", bbox_inches="tight"
)
plt.show()

# Weights

In [ ]:
g = sns.relplot(
    weights.merge(features),
    kind="line",
    x="trainer.representations.layer",
    y="AbsWeight",
    hue="Feature",
    hue_order=features.Feature,
    col="trainer.model_builder.param",
    aspect=1.5,
    height=4.5,
    errorbar="sd",
    marker="o",
    markersize=6,
)
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.set_xlabel("Layer")
    ax.set_ylabel("Absolute Weight")
g.fig.subplots_adjust(wspace=0.05)
ax.set_xticks([1, 3, 6, 9, 12])
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/layers/weights.pdf", bbox_inches="tight")
plt.show()

# Weighted $\tau$ by layer

## Between MLEM and FR-RSA

In [ ]:
weightedtau_weights = weights.pivot(
    index=["Feature", "cv", "trainer.representations.layer"],
    columns=["trainer.model_builder.param"],
    values="AbsWeight",
).reset_index()
weightedtau_weights = (
    weightedtau_weights.groupby(
        ["cv", "trainer.representations.layer"],
    )
    .apply(
        lambda x: stats.weightedtau(x[hue_order[0]], x[hue_order[1]]).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)
weightedtau_weights["Type"] = "Weights"

weightedtau_fis = fi.pivot(
    index=["Feature", "cv", "trainer.representations.layer"],
    columns=["trainer.model_builder.param"],
    values="mean",
).reset_index()
weightedtau_fis = (
    weightedtau_fis.groupby(
        ["cv", "trainer.representations.layer"],
    )
    .apply(
        lambda x: stats.weightedtau(x[hue_order[0]], x[hue_order[1]]).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)
weightedtau_fis["Type"] = "FIs"

weightedtau = pd.concat([weightedtau_weights, weightedtau_fis])

In [ ]:
ax = sns.lineplot(
    weightedtau,
    x="trainer.representations.layer",
    y="Weighted $\\tau$",
    hue="Type",
    style="Type",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_xlabel("Layer")
ax.set_xticks([1, 3, 6, 9, 12])
ax.set_title("Weighted $\\tau$ between\nMLEM and FR-RSA-I", pad=10)
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/layers/weighted_tau_mlem_frrsa.pdf")
plt.show()

## Between FIs and weights

In [ ]:
weightedtau = weights.merge(fi, on=id_cols + ["Feature"])
weightedtau = (
    weightedtau.groupby(
        id_cols,
    )
    .apply(
        lambda x: stats.weightedtau(x.Weight.abs(), x["mean"].abs()).statistic,
        include_groups=False,
    )
    .reset_index(name="Weighted $\\tau$")
)

In [ ]:
ax = sns.lineplot(
    weightedtau,
    x="trainer.representations.layer",
    y="Weighted $\\tau$",
    hue="trainer.model_builder.param",
    hue_order=hue_order,
    style="trainer.model_builder.param",
    markers=True,
    dashes=False,
    errorbar="sd",
)
ax.get_legend().set_title(None)
ax.set_xlabel("Layer")
ax.set_xticks([1, 3, 6, 9, 12])
ax.set_title("Weighted $\\tau$ between\nabsolute weight and FIs", pad=10)
sns.despine(trim=True)
plt.savefig("../../../paper/figs/BERT_RC_mean/layers/weighted_tau_weights_fis.pdf")
plt.show()